In [ ]:
import os
import re
from typing import List, TypedDict, Literal
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langgraph.graph import END, StateGraph, START

from src.ingestion.parser import DocumentIngestionPipeline
from src.retrieval.vector_store import HybridVectorStore
from src.retrieval.reranker import RerankerEngine
from src.agent.llm import get_groq_llm

load_dotenv()

# --- 1. Define Agent State ---
class AgentState(TypedDict):
    question: str
    generation: str
    documents: List[Document]
    retry_count: int
    is_page_query: bool  # Tracks if user asked for a specific page number

# --- 2. Define Pydantic Schemas ---
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

# --- 3. Initialize Shared Singletons ---
llm = get_groq_llm(temperature=0.0)
reranker = RerankerEngine()
vector_store = HybridVectorStore()

# --- 4. Node Definitions ---

def retrieve_node(state: AgentState) -> dict:
    """Retrieves document chunks, supporting single and multi-page requests."""
    print("\n--- 📥 NODE: RETRIEVING & RERANKING ---")
    question = state["question"]
    
    # Extract ALL page numbers mentioned in the prompt (e.g., "page 1 and page 3" -> [1, 3])
    page_matches = re.findall(r'page\s*(?:number)?\s*(\d+)', question.lower())
    page_filters = [int(p) for p in page_matches] if page_matches else []
    is_page_query = len(page_filters) > 0

    all_docs = []
    if is_page_query:
        print(f"🎯 Multi-Page Request Detected for Pages: {page_filters}")
        # Retrieve chunks for each requested page
        for p in page_filters:
            docs = vector_store.hybrid_search(question, top_k=5, page_filter=p)
            all_docs.extend(docs)
        
        # Take the top chunks per page combined
        reranked_docs = all_docs
    else:
        candidate_docs = vector_store.hybrid_search(question, top_k=8)
        reranked_docs = reranker.rerank(question, candidate_docs, top_n=3)
    
    return {
        "documents": reranked_docs, 
        "retry_count": state.get("retry_count", 0),
        "is_page_query": is_page_query
    }

def grade_documents_node(state: AgentState) -> dict:
    """Grades document relevance, skipping checks for direct page requests."""
    print("\n--- 🧐 NODE: EVALUATING DOCUMENT RELEVANCE ---")
    question = state["question"]
    documents = state["documents"]
    is_page_query = state.get("is_page_query", False)
    
    # If user explicitly asked for a page number, keep all retrieved chunks
    if is_page_query and documents:
        print(f"✅ Explicit page requested. Keeping all {len(documents)} page chunks.")
        return {"documents": documents}

    prompt = ChatPromptTemplate.from_template("""
Analyze if the following retrieved document snippet contains relevant information to answer the user's question.
Respond with ONLY 'yes' or 'no'. Do not add any punctuation or extra explanation.

Retrieved Document Snippet:
{document}

Question:
{question}

Is relevant (yes/no):""")

    chain = prompt | llm
    
    filtered_docs = []
    for doc in documents:
        try:
            res = chain.invoke({"document": doc.page_content, "question": question})
            answer = res.content.strip().lower()
            if "yes" in answer:
                filtered_docs.append(doc)
        except Exception as e:
            # Fallback to keeping doc if grading fails
            print(f"⚠️ Grading parse issue, retaining doc by default: {e}")
            filtered_docs.append(doc)
            
    print(f"📊 Filtered {len(filtered_docs)} relevant docs out of {len(documents)}")
    return {"documents": filtered_docs}

def generate_node(state: AgentState) -> dict:
    """Generates a well-structured, coherent answer grounded in retrieved context."""
    print("\n--- 🤖 NODE: GENERATING RESPONSE ---")
    question = state["question"]
    documents = state["documents"]
    
    if not documents:
        return {"generation": "I cannot find any content for the requested query in the uploaded document."}

    context = "\n\n".join([f"[Page {d.metadata.get('page', 'N/A')}]: {d.page_content}" for d in documents])
    
    prompt = ChatPromptTemplate.from_template("""
You are an enterprise AI assistant answering questions based on document context.

Instructions:
1. Synthesize and summarize the provided context into clear, professional paragraphs or bullet points.
2. DO NOT repeat or list individual raw sentences verbatim.
3. Cite page numbers naturally inline in brackets, e.g., [Page 4], at the end of relevant points or sentences.
4. If you cannot answer based on the context, state that clearly.

Context:
{context}

Question:
{question}

Clean Answer:
""")
    
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    return {"generation": response.content}

def transform_query_node(state: AgentState) -> dict:
    """Rewrites query concisely without conversational preamble."""
    print("\n--- 🔄 NODE: REWRITING QUERY FOR BETTER RETRIEVAL ---")
    question = state["question"]
    retry_count = state.get("retry_count", 0) + 1
    
    prompt = ChatPromptTemplate.from_template("""
Rewrite the user's search query to be short, keyword-focused, and optimized for vector retrieval.
OUTPUT ONLY THE REWRITTEN QUERY STRING. DO NOT ADD ANY EXPLANATIONS OR PREAMBLE.

Original Question: {question}

Search Query:""")
    
    chain = prompt | llm
    new_query = chain.invoke({"question": question}).content.strip()
    # Strip any leftover quote marks
    new_query = new_query.replace('"', '').replace("'", "")
    print(f"🔀 Rewritten Query: '{new_query}'")
    return {"question": new_query, "retry_count": retry_count}

# --- 5. Conditional Routing Logic ---

def decide_to_generate(state: AgentState) -> Literal["transform_query", "generate"]:
    """Routes to generation or query transformation."""
    documents = state["documents"]
    retry_count = state.get("retry_count", 0)
    
    if not documents and retry_count < 2:
        print("⚠️ No relevant docs found. Routing to Query Transformation...")
        return "transform_query"
    else:
        print("✅ Relevant docs confirmed. Routing to Generator...")
        return "generate"

# --- 6. Construct LangGraph ---

def build_agent_graph():
    workflow = StateGraph(AgentState)
    
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("grade_documents", grade_documents_node)
    workflow.add_node("generate", generate_node)
    workflow.add_node("transform_query", transform_query_node)
    
    workflow.add_edge(START, "retrieve")
    workflow.add_edge("retrieve", "grade_documents")
    
    workflow.add_conditional_edges(
        "grade_documents",
        decide_to_generate,
        {
            "transform_query": "transform_query",
            "generate": "generate"
        }
    )
    
    workflow.add_edge("transform_query", "retrieve")
    workflow.add_edge("generate", END)
    
    return workflow.compile()


if __name__ == "__main__":
    pdf_path = "data/raw/arxiv.pdf"
    if os.path.exists(pdf_path):
        parser = DocumentIngestionPipeline()
        chunks = parser.process_pdf(pdf_path)
        vector_store.index_documents(chunks)
        
        app = build_agent_graph()
        result = app.invoke({"question": "give me page number 8 content summerization", "retry_count": 0})
        print("\n================ FINAL AGENT RESULT ================")
        print(result["generation"])
        print("====================================================")